# Curriculum Learning (Warm-Up Epoch)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style="whitegrid")

np.random.seed(42)

In [ ]:
n_ad_nc = 200
ad_nc_patients = [f"Subjek_ADNC_{i:03d}" for i in range(1, n_ad_nc + 1)]
ad_nc_labels = np.random.choice(["AD", "NC"], size=n_ad_nc)

n_mci = 200
mci_patients = [f"Subjek_MCI_{i:03d}" for i in range(1, n_mci + 1)]
mci_labels = np.random.choice(["pMCI", "sMCI"], size=n_mci)

# AD vs NC 
loss_ad_nc = np.random.beta(a=1.5, b=5, size=n_ad_nc) * 1.5 

# pMCI vs sMCI 
loss_mci = np.random.beta(a=2.5, b=3, size=n_mci) * 2.0 

df_ad_nc = pd.DataFrame({"Patient_ID": ad_nc_patients, "Class": ad_nc_labels, "Task": "AD vs NC", "Loss_Score": loss_ad_nc})
df_mci = pd.DataFrame({"Patient_ID": mci_patients, "Class": mci_labels, "Task": "pMCI vs sMCI", "Loss_Score": loss_mci})

df_all = pd.concat([df_ad_nc, df_mci]).reset_index(drop=True)
df_all.head(10)

In [ ]:
def categorize_difficulty(loss):
    if loss < 0.3:
        return "Easy"
    elif loss <= 0.8:
        return "Medium"
    else:
        return "Hard"

df_all["Difficulty"] = df_all["Loss_Score"].apply(categorize_difficulty)

display(df_all.groupby(["Task", "Difficulty"]).size().unstack(fill_value=0))

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=df_all, x="Loss_Score", hue="Task", bins=40, kde=True, palette="viridis")
plt.axvline(x=0.3, color='green', linestyle='--', label='Batas Easy (< 0.3)')
plt.axvline(x=0.8, color='red', linestyle='--', label='Batas Medium (0.3 - 0.8)')
plt.title("Distribusi Loss Pasien di Epoch ke-10 (Akhir Warm-Up)")
plt.xlabel("Cross-Entropy Loss (Semakin tinggi = Semakin sulit ditebak)")
plt.ylabel("Jumlah Pasien")
plt.legend()
plt.show()

### Logika Kurikulum (Warm-up 10 Epochs)
Pada **10 epoch pertama (Warm-up)**, kita ingin model membangun fondasi pemahaman fitur yang solid. Model HANYA akan disuapi pasien dari kategori **"Easy"** (Loss < 0.3). 

Setelah epoch 10 terlewati, model baru perlahan-lahan dikenalkan pada data **"Medium"** (misal: di epoch 15), dan pada akhirnya dipaksa mempelajari kasus-kasus ekstrem/sulit **"Hard"** (misal: pMCI yang gambarnya sangat mirip sMCI) di epoch ke-30 hingga akhir.

In [ ]:
# Warm-up Epoch
epoch = 5
if epoch <= 10:
    print(f"--- Epoch {epoch} (Warm-up Phase) ---")
    active_data = df_all[df_all["Difficulty"] == "Easy"]
    print(f"Data aktif: {len(active_data)} sampel termudah.")
else:
    print(f"--- Epoch {epoch} (Scaling Phase) ---")
    
display(active_data.sample(5))

In [ ]:
# Simulasi Data Test & Hasil Prediksi (Klasifikasi Biner: pMCI vs sMCI)
y_test_true = np.random.choice([0, 1], size=100) # 0 = sMCI, 1 = pMCI

y_pred_probs = np.random.beta(a=8, b=2, size=100) * y_test_true + np.random.beta(a=2, b=8, size=100) * (1 - y_test_true)
y_pred = (y_pred_probs > 0.5).astype(int)

print("=== Laporan Klasifikasi Test (pMCI vs sMCI) ===")
print(classification_report(y_test_true, y_pred, target_names=["sMCI", "pMCI"]))

plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test_true, y_pred), annot=True, fmt='d', cmap='Blues', xticklabels=["sMCI", "pMCI"], yticklabels=["sMCI", "pMCI"])
plt.title("Confusion Matrix (Data Test)")
plt.xlabel("Prediksi Model")
plt.ylabel("Data Asli (Ground Truth)")
plt.show()